In [3]:
import os
import glob
import numpy as np
import whisperx
import gc
import subprocess
import torch
from tqdm import tqdm

# --- 1. ตั้งค่าพื้นฐาน ---
NPY_DIR = 'features'
BASE_DIR = '' # เปลี่ยนเป็นพาทเดียวกับสคริปต์ก่อนหน้า
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"
BATCH_SIZE = 48

# --- ตั้งค่าสำหรับงานวิจัย Text2Sign ---
MAX_DURATION_SEC = 12.0  # หั่นประโยคไม่ให้ยาวเกิน 12 วินาที (ป้องกัน Exposure Bias)
SILENCE_THRESH = 1.5     # ถ้าเงียบเกิน 1.5 วินาที ให้ตัดขึ้นประโยคใหม่

def extract_audio(video_path, output_audio_path):
    """สกัดเสียงจากวิดีโอเป็น 16kHz Mono สำหรับ WhisperX"""
    cmd = f"ffmpeg -i {video_path} -vn -acodec pcm_s16le -ar 16000 -ac 1 {output_audio_path} -loglevel quiet -y"
    subprocess.call(cmd, shell=True)

def split_long_segments_by_words(segment, max_duration=12.0, silence_thresh=1.5):
    """ฟังก์ชันสับประโยคยาวๆ ให้สั้นลง โดยอิงจาก Word-Level Alignment"""
    words = segment.get('words', [])
    
    # ถ้าโมเดล align พลาด ไม่มีคำข้างในเลย ให้คืนค่าเดิมกลับไป
    if not words:
        return [{
            'text': segment.get('text', '').strip(),
            'timestamp': (segment.get('start'), segment.get('end')),
            'words': []
        }]

    chunks = []
    current_text = ""
    current_words = []
    chunk_start = None
    last_word_end = None

    for word in words:
        # บางคำที่ Confidence ต่ำ จะไม่มี start/end ให้สะสมคำต่อไป
        if 'start' not in word or 'end' not in word:
            current_text += word.get('word', '')
            current_words.append(word)
            continue

        word_start = word['start']
        word_end = word['end']

        if chunk_start is None:
            chunk_start = word_start

        if last_word_end is not None:
            is_too_long = (word_end - chunk_start) > max_duration
            is_long_pause = (word_start - last_word_end) > silence_thresh

            # ถ้าถึงจุดที่ต้องตัด
            if is_too_long or is_long_pause:
                chunks.append({
                    'text': current_text.strip(),
                    'timestamp': (chunk_start, last_word_end),
                    'words': current_words
                })
                current_text = ""
                current_words = []
                chunk_start = word_start

        current_text += word.get('word', '')
        current_words.append(word)
        last_word_end = word_end

    # เก็บตกส่วนที่เหลือ
    if current_text and chunk_start is not None:
        chunks.append({
            'text': current_text.strip(),
            'timestamp': (chunk_start, last_word_end),
            'words': current_words
        })

    return chunks

def main():
    print(f"🚀 เริ่มต้นซ่อมและจัดระเบียบ Timestamp (Word-Level) ด้วย WhisperX บน {DEVICE}")
    
    # 2. โหลดโมเดล WhisperX และ Alignment
    print("⏳ โหลดโมเดล WhisperX (large-v3)...")
    model = whisperx.load_model("large-v3", DEVICE, compute_type=COMPUTE_TYPE)
    
    model_name = "airesearch/wav2vec2-large-xlsr-53-th"
    print(f"⏳ โหลดโมเดล Alignment ภาษาไทย ({model_name})...")
    align_model, metadata = whisperx.load_align_model(language_code="th", device=DEVICE, model_name=model_name)

    files = glob.glob(os.path.join(NPY_DIR, '*.npy'))
    print(f"📊 พบไฟล์ที่ต้องรันทั้งหมด {len(files)} ไฟล์")

    for file_path in tqdm(files):
        try:
            data = np.load(file_path, allow_pickle=True).item()
            relative_path = data.get('original_path')
            
            if not relative_path:
                print(f"⚠️ ข้าม {file_path}: ไม่พบ original_path")
                continue

            video_path = os.path.join(BASE_DIR, relative_path)
            if not os.path.exists(video_path):
                print(f"⚠️ ข้าม {file_path}: หาไฟล์วิดีโอไม่เจอ {video_path}")
                continue

            temp_audio = f"/dev/shm/temp_audio_fix.wav"
            
            # --- กระบวนการ WhisperX ---
            extract_audio(video_path, temp_audio)
            audio = whisperx.load_audio(temp_audio)
            
            # 1. ถอดเสียง
            result = model.transcribe(audio, batch_size=BATCH_SIZE, language="th")
            
            # 2. ทำ Alignment 
            result_aligned = whisperx.align(result["segments"], align_model, metadata, audio, DEVICE, return_char_alignments=False)

            # --- 3. หั่นย่อยด้วยลอจิก Word-Level 🌟 ---
            new_chunks = []
            full_text = ""
            
            for segment in result_aligned["segments"]:
                # เรียกใช้ฟังก์ชันสับประโยค
                sub_chunks = split_long_segments_by_words(segment, max_duration=MAX_DURATION_SEC, silence_thresh=SILENCE_THRESH)
                
                for chunk in sub_chunks:
                    new_chunks.append({
                        'timestamp': chunk['timestamp'],
                        'text': chunk['text'],
                        'words': chunk['words'] # เก็บ words ไว้เผื่อใช้ในอนาคต
                    })
                    full_text += chunk['text'] + " "

            # --- อัปเดตข้อมูลทับลงไป ---
            data['asr'] = full_text.strip()
            data['asr_chunks'] = new_chunks
            
            # เซฟทับไฟล์เดิม (keypoints ปลอดภัยหายห่วง)
            np.save(file_path, data, allow_pickle=True)
            
            if os.path.exists(temp_audio):
                os.remove(temp_audio)

        except Exception as e:
            print(f"\n❌ ประมวลผลไฟล์ {os.path.basename(file_path)} ไม่สำเร็จ: {e}")

    print("🎉 อัปเกรด Dataset เสร็จสมบูรณ์! ข้อมูลถูกหั่นด้วย Word-Level Alignment พร้อมสำหรับงานวิจัยแล้วครับ!")

if __name__ == "__main__":
    main()

🚀 เริ่มต้นซ่อมและจัดระเบียบ Timestamp (Word-Level) ด้วย WhisperX บน cuda
⏳ โหลดโมเดล WhisperX (large-v3)...


/home/rehab/research/text2sign/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-04 03:13:58 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-05-04 03:13:58 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint .venv/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


⏳ โหลดโมเดล Alignment ภาษาไทย (airesearch/wav2vec2-large-xlsr-53-th)...


...

📊 พบไฟล์ที่ต้องรันทั้งหมด 281 ไฟล์


  0%|          | 0/281 [00:00<?, ?it/s]/home/rehab/research/text2sign/.venv/lib/python3.12/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
 69%|██████▊   | 193/281 [17:36<08:04,  5.51s/it]

2026-05-04 03:31:44 - whisperx.alignment - WARNING - Failed to align segment ("สวัสดีครับ"): backtrack failed, resorting to original


100%|██████████| 281/281 [25:45<00:00,  5.50s/it]

🎉 อัปเกรด Dataset เสร็จสมบูรณ์! ข้อมูลถูกหั่นด้วย Word-Level Alignment พร้อมสำหรับงานวิจัยแล้วครับ!
